In [ ]:
import numpy as np

def centered_finite_difference(objective, theta, step=1e-5):
    theta = np.asarray(theta, dtype=float)
    grad = np.zeros_like(theta)

    for j in range(theta.size):
        shift = np.zeros_like(theta)
        shift[j] = step
        plus = objective(theta + shift)
        minus = objective(theta - shift)
        grad[j] = (plus - minus) / (2.0 * step)

    return grad

def gradient_descent(objective, grad_fn, theta0, lr=0.05, steps=100):
    theta = np.asarray(theta0, dtype=float).copy()
    history = []

    for _ in range(steps):
        value = float(objective(theta))
        grad = np.asarray(grad_fn(theta), dtype=float)
        history.append((value, np.linalg.norm(grad)))
        theta = theta - lr * grad

    return theta, np.array(history)

def check_descent_step(objective, theta, grad, lr):
    before = float(objective(theta))
    after = float(objective(theta - lr * grad))
    return before, after, after <= before

def cosine_similarity(a, b, eps=1e-12):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom < eps:
        return np.nan
    return float(np.dot(a, b) / denom)

# Example: compare an analytic gradient with a numerical check.
def quadratic(theta):
    target = np.array([1.0, -2.0, 0.5])
    residual = theta - target
    return 0.5 * np.dot(residual, residual)

def quadratic_grad(theta):
    target = np.array([1.0, -2.0, 0.5])
    return theta - target

theta = np.array([0.0, 0.0, 0.0])
analytic = quadratic_grad(theta)
numeric = centered_finite_difference(quadratic, theta)

print("analytic:", analytic)
print("numeric: ", numeric)
print("cosine: ", cosine_similarity(analytic, numeric))
print("descent: ", check_descent_step(quadratic, theta, analytic, lr=0.1))


In [ ]:
# Runtime checks for the Appendix III listing.
theta = np.array([0.0, 0.0, 0.0])
analytic = quadratic_grad(theta)
numeric = centered_finite_difference(quadratic, theta)

assert np.allclose(analytic, [-1.0, 2.0, -0.5])
assert np.allclose(numeric, analytic, atol=1e-8)
assert np.isclose(cosine_similarity(analytic, numeric), 1.0)

before, after, decreased = check_descent_step(quadratic, theta, analytic, lr=0.1)
assert np.isclose(before, 2.625)
assert np.isclose(after, 2.12625)
assert decreased

opt_theta, history = gradient_descent(
    quadratic,
    quadratic_grad,
    theta0=np.array([0.0, 0.0, 0.0]),
    lr=0.1,
    steps=120,
)

assert history.shape == (120, 2)
assert history[-1, 0] < history[0, 0]
assert history[-1, 1] < history[0, 1]
assert np.allclose(opt_theta, [1.0, -2.0, 0.5], atol=1e-5)
assert np.isnan(cosine_similarity([0.0, 0.0], [1.0, 2.0]))

print("Appendix III checks passed.")
